In [11]:
import pandas as pd
import ast
from pathlib import Path

**DataFrame** class: 2D tabular data structure, 2 label system. Columns and Indexes (rows). Each column has a name, and has data as a Series object. Each row has an index, one label per row, label can be any hashable type, they are immutable, ordered arrays of row labels.

In [ ]:
df = pd.DataFrame({
    "patient": ["p1", "p2", "p3", "p1", "p2", "p3"],
    "organ":   ["liver", "liver", "liver", "heart", "heart", "heart"],
    "version": ["v1", "v1", "v1", "v2", "v2", "v2"],
    "metric":  ["chamfer"] * 6,
    "value":   [1.2, 1.0, 1.1, 0.9, 1.1, 1.0],
}) # create with ignore_index=True if not wanting index to appear alongside rows

# OSS: There is NO row for:
# (p1, heart, v1)
# (p2, heart, v1)
# (p3, heart, v1)
# (p1, liver, v2)
# (p2, liver, v2)
# (p3, liver, v2)

# Those combinations simply do not exist.

In [19]:
df

,patient,organ,version,metric,value
0,p1,liver,v1,chamfer,1.2
1,p2,liver,v1,chamfer,1.0
2,p3,liver,v1,chamfer,1.1
3,p1,heart,v2,chamfer,0.9
4,p2,heart,v2,chamfer,1.1
5,p3,heart,v2,chamfer,1.0


In [20]:
df["organ"] # returns a Series

0    liver
1    liver
2    liver
3    heart
4    heart
5    heart
Name: organ, dtype: object

In [21]:
df[["organ", "value"]] # returns a DataFrame

,organ,value
0,liver,1.2
1,liver,1.0
2,liver,1.1
3,heart,0.9
4,heart,1.1
5,heart,1.0


In [22]:
df.loc[0] # row with index LABEL 0
df.iloc[0] # actual first row by position

patient         p1
organ        liver
version         v1
metric     chamfer
value          1.2
Name: 0, dtype: object

Analyze in LONG format: each row = one observation, variables are columns, repeated measures create more rows --> native format for statistics

- *groupby* : splits rows into groups, applies a function to each group, combines results in a new object. GroupBy object stores group keys --> row indeces

- *agg* : aggregation, reduces each group to one row, so number of groups = number of rows, and group keys become row labels (index). If not wanted, either call .reset_index(), or create the groupby object with the flag as_index=False, to have group keys already as columns

EX: mean per organ per version

In [23]:
df.groupby(["version","organ"])["value"].mean()

version  organ
v1       liver    1.1
v2       heart    1.0
Name: value, dtype: float64

In [24]:
df.groupby(["version","organ"])["value"].mean().reset_index()

,version,organ,value
0,v1,liver,1.1
1,v2,heart,1.0


In [25]:
df.groupby(["version","organ"], as_index=False)["value"].mean()

,version,organ,value
0,v1,liver,1.1
1,v2,heart,1.0


EX: std per organ

In [26]:
df.groupby("organ")["value"].std()

organ
heart    0.1
liver    0.1
Name: value, dtype: float64

FILTER: with query, returns a dataframe

In [27]:
df.query("organ == 'liver' and version == 'v1' ")

,patient,organ,version,metric,value
0,p1,liver,v1,chamfer,1.2
1,p2,liver,v1,chamfer,1.0
2,p3,liver,v1,chamfer,1.1


WIDE format: one variable's value is spread across columns. Wide format does not scale when adding new data and analyzing again. It is useful for comparing: for example having each row = same patient + organ, each column = a version is impossible to get directly from long format without pivoting, and to compare for example versions (like taking a difference in values for example) is hard in long more, trivial in wide mode. 

Another example: paired statistical tests, they require (x1_i, x2_i) which is exactly what wide format gives.

pivot / pivot_table : goes from long to wide format

In [ ]:
wide = df.pivot_table(
    index=["patient", "organ"],
    columns="version",
    values="value"
)
# That means:
# Rows = all observed (patient, organ) pairs
# Columns = all observed version values
# Cell value = value for that exact combination
# So pandas tries to build: patient	organ	v1	v2
wide

version         v1   v2
patient organ          
p1      heart  NaN  0.9
        liver  1.2  NaN
p2      heart  NaN  1.1
        liver  1.0  NaN
p3      heart  NaN  1.0
        liver  1.1  NaN

In [30]:
wide["delta_v2_v1"] = wide["v2"] - wide["v1"]
wide

version         v1   v2  delta_v2_v1
patient organ                       
p1      heart  NaN  0.9          NaN
        liver  1.2  NaN          NaN
p2      heart  NaN  1.1          NaN
        liver  1.0  NaN          NaN
p3      heart  NaN  1.0          NaN
        liver  1.1  NaN          NaN

To go back to analysis, so back to long format, use MELT

In [31]:
long_again = wide.reset_index().melt(
    id_vars=["patient", "organ"],
    var_name="version",
    value_name="value"
)
long_again

,patient,organ,version,value
0,p1,heart,v1,NaN
1,p1,liver,v1,1.2
2,p2,heart,v1,NaN
3,p2,liver,v1,1.0
4,p3,heart,v1,NaN
5,p3,liver,v1,1.1
6,p1,heart,v2,0.9
7,p1,liver,v2,NaN
8,p2,heart,v2,1.1
9,p2,liver,v2,NaN


Intuition:

LONG: \
p1 liver v1 1.2 \
p1 liver v2 1.1 \
p2 liver v1 1.0 \
p2 liver v2 0.95 

WIDE: \
p1 liver  v1=1.2  v2=1.1 \
p2 liver  v1=1.0  v2=0.95 

Use LONG format to:

- filter

- group

- aggregate

- visualize distributions

Use WIDE format to:

- compare conditions

- compute deltas

- do paired statistics

concatenate several dataframes: dfs = [df1, df2, ...] --> pd.concat(dfs, ignore_index=True)

Using parquet instead of csv: faster loading, can load just some of the file instead with csv it has to be read whole.

With pandas, use to_parquet() instead of to_csv(), load with read_parquet() instead of read_csv() 

So maybe use parquet while experimenting, then final results saved as csv for readability / sharing ...

parquet --> dataframe storage format
csv --> text interchange format

# LOAD FROM FILE 
Keep "metric" in every groupby, so to never average or apply any aggregation to different kinds of metrics like chamfer and LDDMM

In [7]:
name = "version_114-LDDMM.csv"
df = pd.read_csv("results/metrics/" + name)

In [8]:
organ_stats = (
    df.groupby(["metric", "organ"])["value"]
      .agg(["mean", "median", "std", "min", "max", "count"])
      .reset_index()
)
organ_stats

,metric,organ,mean,median,std,min,max,count
0,LDDMM,epicardium,1991609.550,2076361.100,436406.168470,1432494.50,2381221.5,4
1,LDDMM,la_endo,402685.415,417246.155,142664.374660,219872.25,556377.1,4
2,LDDMM,ra_endo,394414.485,390714.970,119464.232795,290490.00,505738.0,4


In [9]:
patient_stats = (
    df.groupby(["metric", "patient"])["value"]
      .mean()
      .reset_index(name="mean_value")
)
patient_stats

,metric,patient,mean_value
0,LDDMM,AF001,1.080280e+06
1,LDDMM,AF009_P2R,1.076444e+06
2,LDDMM,LEU_NORM_0032,9.139359e+05
3,LDDMM,LEU_NORM_F004,6.476189e+05


In [21]:
df.groupby(["metric", "organ"])["value"].agg(
    mean="mean",
    median="median",
    std="std",
)

mean    median       std
metric  organ                                   
chamfer epicardium  0.005080  0.005073  0.000101
        la_endo     0.004609  0.004600  0.000102
        ra_endo     0.004500  0.004385  0.000277

In [22]:
df.groupby(["version", "metric", "organ"])["value"].mean()

version  metric   organ     
114      chamfer  epicardium    0.005080
                  la_endo       0.004609
                  ra_endo       0.004500
Name: value, dtype: float64

In [ ]:
metric_summary = (
    df.groupby("metric")["value"]
      .agg(["mean", "median", "std"])
)
metric_summary

,mean,median,std
metric,,,
LDDMM,929569.816667,497717.03,822575.37369


In [ ]:
def parse_value(x):
    if isinstance(x, str) and x.startswith("["):
        return ast.literal_eval(x)[0]
    return float(x)

dfs = []

for csv_path in Path("results/metrics").glob("version_*.csv"):
    # extract version from filename, e.g. version_89.csv → 89
    version = csv_path.stem.split("-")[0].split("_")[-1]

    df = pd.read_csv(csv_path)
    df["version"] = int(version)
    df["value"] = df["value"].apply(parse_value)

    dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)
# all_df["version"].unique() # inspect which versions are there

array([100,  89, 114])

In [20]:
# mean per organ per version
all_df.groupby(["version", "metric", "organ"])["value"].mean() #.reset_index()

version  metric   organ     
89       LDDMM    epicardium    1.991610e+06
                  la_endo       4.026854e+05
                  ra_endo       3.944145e+05
         chamfer  epicardium    5.078220e-03
                  la_endo       4.615072e-03
                  ra_endo       4.502563e-03
100      LDDMM    epicardium    1.991610e+06
                  la_endo       4.026854e+05
                  ra_endo       3.944145e+05
         chamfer  epicardium    5.072637e-03
                  la_endo       4.613504e-03
                  ra_endo       4.484059e-03
114      LDDMM    epicardium    1.991610e+06
                  la_endo       4.026854e+05
                  ra_endo       3.944145e+05
         chamfer  epicardium    5.079981e-03
                  la_endo       4.608905e-03
                  ra_endo       4.500151e-03
Name: value, dtype: float64

Hierarchy:

Version \
 └─ Metric \
     └─ Organ \
         └─ Patient

Per-organ, aggregated over patients. Average error for each organ across all patients

In [23]:
df.groupby(["version", "metric", "organ"])["value"].agg(
    mean="mean",
    median="median",
    std="std",
    q95=lambda x: x.quantile(0.95),
)

mean    median       std       q95
version metric  organ                                             
114     chamfer epicardium  0.005080  0.005073  0.000101  0.005191
                la_endo     0.004609  0.004600  0.000102  0.004721
                ra_endo     0.004500  0.004385  0.000277  0.004838

Per-patient, aggregated over organs. detect global failures, organs are weighted equally here... How well did the method do on this patient overall?

In [24]:
df.groupby(["version", "metric", "patient"])["value"].mean()

version  metric   patient      
114      chamfer  AF001            0.004655
                  AF009_P2R        0.004713
                  LEU_NORM_0032    0.004918
                  LEU_NORM_F004    0.004633
Name: value, dtype: float64

Global summary (all patients, all organs), within one metric

In [25]:
df.groupby(["version", "metric"])["value"].agg(
    mean="mean",
    std="std",
    q95=lambda x: x.quantile(0.95),
)

,,mean,std,q95
version,metric,,,
114,chamfer,0.00473,0.000309,0.005149


# TWO-versions comparisons

patients are PAIRED, in the sense that for a given patient + organ, you compute

Chamfer (GT vs recon) for version A

Chamfer (GT vs recon) for version B

LDDMM for version A

LDDMM for version B

So for the same patient:

Patient AF001
  Epicardium:
    v88 → chamfer = 0.0051
    v89 → chamfer = 0.0048

These two numbers are paired because:

- same anatomy

- same GT

- same organ

- only the method/version changed

That pairing is extremely valuable — it removes patient-to-patient variability.

In [26]:
def parse_value(x):
    if isinstance(x, str) and x.startswith("["):
        return ast.literal_eval(x)[0]
    return float(x)

dfs = []

for csv_path in Path("results/metrics").glob("version_*.csv"):
    # extract version from filename, e.g. version_89.csv → 89
    version = csv_path.stem.split("-")[0].split("_")[-1]

    df = pd.read_csv(csv_path)
    df["version"] = int(version)
    df["value"] = df["value"].apply(parse_value)

    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

In [29]:
# # comparing means without using patient pairing.
# df.groupby(["version", "metric", "organ"])["value"].mean()

# Now each value is “Improvement for this patient & organ” :
pivot = df.pivot_table(
    index=["metric", "organ", "patient"],
    columns="version",
    values="value"
)

delta = pivot[89] - pivot[114]
# each row in delta: Error(version 89) − Error(version 114) for a specific (metric, organ, patient)
# Negative = improvement (lower error). This is the paired improvement per patient.
delta

metric   organ       patient      
LDDMM    epicardium  AF001            0.000000e+00
                     AF009_P2R        0.000000e+00
                     LEU_NORM_0032    0.000000e+00
                     LEU_NORM_F004    0.000000e+00
         la_endo     AF001            0.000000e+00
                     AF009_P2R        0.000000e+00
                     LEU_NORM_0032    0.000000e+00
                     LEU_NORM_F004    0.000000e+00
         ra_endo     AF001            0.000000e+00
                     AF009_P2R        0.000000e+00
                     LEU_NORM_0032    0.000000e+00
                     LEU_NORM_F004    0.000000e+00
chamfer  epicardium  AF001            2.406305e-06
                     AF009_P2R       -1.007521e-05
                     LEU_NORM_0032   -3.739483e-06
                     LEU_NORM_F004    4.365601e-06
         la_endo     AF001            1.709800e-05
                     AF009_P2R        8.737353e-07
                     LEU_NORM_0032    1.539626e

then summarize deltas

Grouping all patient-level deltas

Separately for each metric

Separately for each organ

Then summarizing those deltas across patients

Meaninig:

mean = Average improvement across patients, ex mean = -0.00021 --> “On average, version 89 reduces Chamfer error by 0.00021 for epicardium.”

median = −0.00018 --> For a typical patient, the improvement is about −0.00018
q95 = +0.00004 “For 95% of patients, performance did not degrade by more than 0.00004.” --> Mean can look good But q95 tells you whether some patients got much worse

EXAMPLE:

metric   organ        mean      median     q95

.................................................

chamfer  epicardium  -0.00021   -0.00018   +0.00004

chamfer  la_endo     -0.00015   -0.00012   +0.00002

chamfer  ra_endo     -0.00019   -0.00017   +0.00001

then interpretation is:
“Version 89 improves Chamfer distance across all organs, with mean reductions between 3–6%. Performance degradation is rare, with 95% of patients showing no worse than a 0.00004 increase.”
That is very strong evidence.

“Performance differences between versions were computed on a per-patient and per-organ basis, and summarized using the mean, median, and 95th percentile of the paired error differences.”

“For each organ and metric, performance differences between two versions were evaluated using paired per-patient error differences. Summary statistics (mean, median, and 95th percentile) of these differences were reported.”

In [28]:
delta.groupby(["metric", "organ"]).agg(
    mean="mean",
    median="median",
    q95=lambda x: x.quantile(0.95),
)

mean        median       q95
metric  organ                                       
LDDMM   epicardium  0.000000  0.000000e+00  0.000000
        la_endo     0.000000  0.000000e+00  0.000000
        ra_endo     0.000000  0.000000e+00  0.000000
chamfer epicardium -0.000002 -6.665892e-07  0.000004
        la_endo     0.000006  8.134996e-06  0.000017
        ra_endo     0.000002 -1.910355e-06  0.000024